[Langgraph Superviser](https://reference.langchain.com/python/langgraph-supervisor)

Use a Single Agent if:
  • The task is one coherent role
  • Tool count is small (<15)
  • No real specialization needed
  • Latency and simplicity matter

Use a Supervisor if:
  • Multiple specialized capabilities
  • Need controlled, debuggable routing
  • Want one place to change orchestration logic
  • This is your default production choice

Use Supervisor-as-Tools if:
  • You want native tool-calling for routing
  • Specialists should be isolated subtasks
  • Supervisor should retain final control of user-facing output

Use Handoffs / Swarm if:
  • Different agents own different parts of the conversation
  • The active agent should change over time

Use Hierarchical if:
  • You have many agents (10+)
  • Work decomposes into sub-teams

Use Custom LangGraph if:
  • Workflow structure is known
  • You need deterministic control with branching
  • You need retries, validators, checkpoints, human review


### Side-by-side comparison

| Pattern | Control | State sharing | Debuggability | Flexibility | Latency | Best for |
|---|---|---|---|---|---|---|
| **Single Agent** | Implicit (LLM loop) | Single context | Easy | Low | Lowest | Focused single-role tasks |
| **Network** | Distributed | Shared, all agents see all | Hard | Highest | Variable | Open-ended exploration (rare in prod) |
| **Supervisor** | Centralized | Shared via supervisor | Easy | Medium | Medium | Most production systems |
| **Supervisor-as-Tools** | Centralized | Isolated subtasks | Easy | Medium | Medium | Clean separation, native tool-calling |
| **Hierarchical** | Multi-level | Per-team + global | Medium | High | High | Large org-style workflows |
| **Custom** | Explicit (graph) | Designed | Easy (you wrote it) | High in theory, fixed in practice | Tunable | Mature production workflows |

Picking a pattern decides the *topology* of who talks to whom. But there are several distinct *mechanisms* through which agents actually exchange information. Most production systems mix two or three.

| Style | Mechanism | When it shines | What we use it for in this notebook |
|---|---|---|---|
| **Shared state** | Agents read/write fields on a typed graph state object | Pipelines where each step contributes a structured artifact | The 3-agent research team — Planner writes `plan`, Researcher writes `research_notes`, Writer reads both |
| **Message log** | Agents append to a running list of messages | Audit trails, conversational continuity, traceability | The `messages` field — every agent leaves a breadcrumb so we can reconstruct what happened |
| **Tool call** | One agent invokes another as a tool with structured args | Clean isolation; supervisor wants final control | The Supervisor-as-Tools pattern (2D) |
| **Handoff** | Active agent calls a `transfer_to_X` tool and control sticks with the new agent | Conversational ownership: triage → tech support → escalation | The swarm pattern (Section 8) |

> **Instructor Note:** "Which communication style?" is just as important a design question as "which topology?". A supervisor pattern with **shared state + structured tool returns** is very different in practice from a supervisor pattern that uses **only a message log** — the first is debuggable, the second is fragile.


| Failure | Symptom | Fix |
|---|---|---|
| **Wrong agent selected** | Researcher runs when Writer was needed | Better supervisor prompt; structured decision schema; rule-based supervisor for trivial cases |
| **Repeated work** | Same agent runs twice with same input | Attempt counter; cache by state hash |
| **Lost context** | Agent B doesn't know what Agent A produced | State design — make sure A's output is in the state field B reads |
| **State overwritten** | Field set by A is gone after B runs | Don't have B return that field; use LangGraph reducers |
| **Agents disagree** | Two agents return contradictory facts | Add a Critic; share evidence pool; reduce autonomy |
| **Infinite loop** | Same routing decision over and over | `recursion_limit`; explicit termination; visit counts |
| **Search returned nothing** | Notes empty, Writer hallucinates | `safe_search` with try/except; defensive empty-notes path; mark uncertainty |
| **Cost explosion** | $$$ per request | Smaller routing model; fewer hops; cache plans |
| **Latency explosion** | 30s per request | Parallelize; trim context per agent |


### Optimization targets

| Component | Cost risk | Latency risk | Optimization |
|---|---|---|---|
| Planner | Low | Low | Small/cheap model — planning is short and structured |
| Researcher | **High** (search calls + long prompts) | **High** | Limit search calls per sub-question; compress snippets |
| Writer | Medium-high | Medium | Compress notes before passing in; use a strong model only here |
| Supervisor | Medium | Medium-high | **Use rule-based whenever possible** — see §7.4 |
| Critic | Medium | Medium | Run only when needed; skip for low-risk outputs |

### Rules of thumb

1. **Use the smallest model that works for routing.** Routing is mostly classification.
2. **Use rule-based supervisors when you can.** Section 7.4 dropped supervisor cost to zero.
3. **Cache the plan** if your queries repeat or are similar.
4. **Compress between agents.** The Researcher should produce *notes*, not transcripts.
5. **Parallelize where you can.** Sub-question searches are independent.
6. **Set hard caps.** `recursion_limit`, `MAX_RESEARCHER_ATTEMPTS`, `MAX_CRITIC_ROUNDS`.


1. **Single agents are powerful.** We built one in Section 5 and it produced a real report. Don't reach for multi-agent until that's genuinely failing.
2. **Six patterns cover most needs**: Single Agent, Network, Supervisor, Supervisor-as-Tools, Hierarchical, Custom. **Supervisor is the right default for production.**
3. **Communication style matters as much as topology.** Shared state + structured outputs = debuggable. Free-form messages = fragile.
4. **State is the spine.** Type it. Validate it. Keep agent outputs structured (Pydantic) so the next agent can rely on the contract.
5. **LangGraph gives you nodes + edges + state.** Sequential graphs are deterministic; supervisor graphs trade LLM calls for flexibility.
6. **Sources are first-class.** A research system without grounded citations isn't done.
7. **Loop protection is non-negotiable.** Attempt counters, recursion limits, hard caps.
8. **Debug with structured outputs, state snapshots, recursion limits, safe tool wrappers, and tracing.**
9. **Cost is real.** Routing models small; synthesis models large; compress between agents; set hard caps.